In [1]:
import sys, pandas as pd

print(pd.__version__, sys.prefix)

3.0.6 /Users/hyunjunson/study/data-eng/week3/.venv


In [2]:
# make_orders.py — 3주차 공용 실습 데이터 생성
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n = 1000

orders = pd.DataFrame({
    'order_id': np.arange(1, n + 1),
    'customer_id': rng.integers(1, 201, n),
    'category': rng.choice(['식품', '가전', '의류', '도서'], n, p=[0.4, 0.2, 0.3, 0.1]),
    'price': rng.integers(1, 200, n) * 1000,
    'quantity': rng.integers(1, 6, n),
    'order_date': pd.to_datetime('2026-06-01')
        + pd.to_timedelta(rng.integers(0, 60, n), unit='D'),
})

# 품질 문제를 일부러 심는다 — 3-5 품질 리포트의 재료가 된다
orders.loc[rng.choice(n, 30, replace=False), 'price'] = np.nan      # 결측 30건
orders.loc[rng.choice(n, 20, replace=False), 'category'] = ' 식품 '  # 공백 오염
orders = pd.concat([orders, orders.sample(15, random_state=42)])     # 중복 15건
orders['order_date'] = orders['order_date'].astype(str)              # 날짜를 문자열로

orders.to_csv('orders.csv', index=False, encoding='utf-8')
print(len(orders), '행 저장 완료')


1015 행 저장 완료


In [3]:
# explore.py — 3~6단계를 이 파일에 이어 붙입니다
import pandas as pd

df = pd.read_csv('orders.csv')

print(df.shape)          # (행, 열)
print(df.dtypes)         # 열마다 타입
print(df.head(3))        # 앞 3줄
df.info()                # 결측 수까지 한눈에 (print 로 감싸면 끝에 None 이 한 줄 더 찍힙니다)
print(df.describe())     # 숫자 열의 기초 통계


(1015, 6)
order_id         int64
customer_id      int64
category           str
price          float64
quantity         int64
order_date         str
dtype: object
   order_id  customer_id category     price  quantity  order_date
0         1           18       의류  155000.0         2  2026-06-12
1         2          155       의류  134000.0         5  2026-06-03
2         3          131       도서  186000.0         1  2026-07-08
<class 'pandas.DataFrame'>
RangeIndex: 1015 entries, 0 to 1014
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   order_id     1015 non-null   int64  
 1   customer_id  1015 non-null   int64  
 2   category     1015 non-null   str    
 3   price        985 non-null    float64
 4   quantity     1015 non-null   int64  
 5   order_date   1015 non-null   str    
dtypes: float64(1), int64(3), str(2)
memory usage: 63.6 KB
          order_id  customer_id          price     quantity
count  1015.000000  1015

In [4]:
# 4단계 — explore.py 끝에 이어 붙입니다
# price는 정수만 넣었는데 float64입니다
print('price dtype :', df['price'].dtype)
print('quantity dtype:', df['quantity'].dtype)

# 이유는 결측(NaN) 때문입니다
print('price 결측 :', df['price'].isna().sum())
print('quantity 결측:', df['quantity'].isna().sum())

# 값 몇 개를 보면 1000이 1000.0으로 보입니다
print(df['price'].head(3).tolist())

price dtype : float64
quantity dtype: int64
price 결측 : 30
quantity 결측: 0
[155000.0, 134000.0, 186000.0]


In [5]:
# 5단계 — explore.py 끝에 이어 붙입니다
df.to_parquet('orders_raw.parquet', index=False)

import os
csv_size = os.path.getsize('orders.csv')
pq_size = os.path.getsize('orders_raw.parquet')
print(f'CSV    : {csv_size:,} bytes')
print(f'Parquet: {pq_size:,} bytes  ({pq_size / csv_size:.0%})')


CSV    : 36,244 bytes
Parquet: 15,092 bytes  (42%)


In [6]:
# 6단계 — explore.py 끝에 이어 붙입니다
# Parquet — 두 열만 읽습니다
slim = pd.read_parquet('orders_raw.parquet', columns=['category', 'price'])
print(slim.shape)
print(slim.columns.tolist())

# CSV — usecols로 지정해도 파일 전체는 읽어야 합니다.

slim_csv = pd.read_csv('orders.csv', usecols=['category', 'price'])
print(slim_csv.shape)

(1015, 2)
['category', 'price']
(1015, 2)


In [7]:
# transform.py — 1~8단계를 이 파일에 이어 붙입니다
import pandas as pd
df = pd.read_csv('orders.csv')

# 열 하나 → Series
print(type(df['price']))

# 열 여러 개 → DataFrame (대괄호가 두 겹인 것에 주의)
print(type(df[['order_id', 'price']]))

# 위치로 — iloc (끝 제외)

print(df.iloc[0:3])

# 이름표로 — loc (끝 포함)
print(df.loc[0:2])

<class 'pandas.Series'>
<class 'pandas.DataFrame'>
   order_id  customer_id category     price  quantity  order_date
0         1           18       의류  155000.0         2  2026-06-12
1         2          155       의류  134000.0         5  2026-06-03
2         3          131       도서  186000.0         1  2026-07-08
   order_id  customer_id category     price  quantity  order_date
0         1           18       의류  155000.0         2  2026-06-12
1         2          155       의류  134000.0         5  2026-06-03
2         3          131       도서  186000.0         1  2026-07-08


In [8]:
# 2단계 — transform.py 끝에 이어 붙입니다
# 조건 필터 — 불리언 인덱싱
food = df[df['category'].str.strip() == '식품']
big = df[(df['price'] >= 100000) & (df['quantity'] >= 3)]
print('식품 주문:', len(food), '건 / 고액·다수량:', len(big), '건')

# loc — 행 조건과 열 목록을 한 번에
sub = df.loc[df['customer_id'].isin([1, 2, 3]),
             ['order_id', 'customer_id', 'price']]
print(sub.head(3))

식품 주문: 416 건 / 고액·다수량: 275 건
     order_id  customer_id     price
103       104            2   88000.0
302       303            2  120000.0
380       381            2  166000.0


In [9]:
# 2단계(일부러 실패) — transform.py 끝에 이어 붙입니다
# strip을 빼면 몇 건이 새는지 확인해 봅니다
without_strip = df[df['category'] == '식품']
with_strip = df[df['category'].str.strip() == '식품']
print('strip 없이:', len(without_strip))
print('strip 하고:', len(with_strip))
print('놓친 건수 :', len(with_strip) - len(without_strip))

strip 없이: 396
strip 하고: 416
놓친 건수 : 20


In [11]:
# 3단계 — transform.py 끝에 이어 붙입니다
# 어떤 값들이 들어 있나
print(df['category'].unique())
print(df['category'].nunique(), '가지')


# 각각 몇 건인가 — 오염이 눈에 보입니다
# print(df['category'].value_counts())

<ArrowStringArray>
['의류', '도서', '가전', ' 식품 ', '식품']
Length: 5, dtype: str
5 가지


In [12]:
print(df['category'].value_counts())

category
식품      396
의류      284
가전      203
도서      112
 식품      20
Name: count, dtype: int64


In [17]:
# 4단계 — transform.py 끝에 이어 붙입니다
df['category'] = df['category'].str.strip()
df['order_date'] = pd.to_datetime(df['order_date'])
df = df.assign(amount=df['price'] * df['quantity'])

print(df.dtypes)
print('6월 첫 주 주문:', len(df[df['order_date'] < '2026-06-08']), '건')
print('카테고리:', df['category'].nunique(), '가지')

order_id                int64
customer_id             int64
category                  str
price                 float64
quantity                int64
order_date     datetime64[us]
amount                float64
dtype: object
6월 첫 주 주문: 103 건
카테고리: 4 가지


In [18]:
# 5단계 — transform.py 끝에 이어 붙입니다
# datetime이 되고 나면 .dt로 부품을 꺼낼 수 있습니다
df['year'] = df['order_date'].dt.year
df['month'] = df['order_date'].dt.month
df['weekday'] = df['order_date'].dt.day_name()

print(df[['order_date', 'year', 'month', 'weekday']].head(3))
print(df['weekday'].value_counts())

  order_date  year  month    weekday
0 2026-06-12  2026      6     Friday
1 2026-06-03  2026      6  Wednesday
2 2026-07-08  2026      7  Wednesday
weekday
Thursday     173
Monday       158
Tuesday      148
Saturday     139
Sunday       138
Friday       130
Wednesday    129
Name: count, dtype: int64


In [19]:
# 6단계 — transform.py 끝에 이어 붙입니다
print('결측 현황:')
print(df.isna().sum())

# 방법 1 — 결측 행을 버린다
print('버리면:', len(df.dropna(subset=['price'])), '행')

# 방법 2 — 대표값으로 채운다
filled = df['price'].fillna(df['price'].median())
print('중앙값으로 채우면 결측:', filled.isna().sum())

# 방법 3 — 그대로 두고 집계에서 제외 (Pandas 기본 동작)
print('평균(결측 자동 제외):', round(df['price'].mean()))

결측 현황:
order_id        0
customer_id     0
category        0
price          30
quantity        0
order_date      0
amount         30
year            0
month           0
weekday         0
dtype: int64
버리면: 985 행
중앙값으로 채우면 결측: 0
평균(결측 자동 제외): 100346


In [20]:
# 7단계 — transform.py 끝에 이어 붙입니다
df.to_parquet('orders_step1.parquet', index=False)

check = pd.read_parquet('orders_step1.parquet')
print('CSV로 읽은 날짜 타입   :', pd.read_csv('orders.csv')['order_date'].dtype)
print('Parquet로 읽은 날짜 타입:', check['order_date'].dtype)
print('저장된 열:', check.columns.tolist())

CSV로 읽은 날짜 타입   : str
Parquet로 읽은 날짜 타입: datetime64[us]
저장된 열: ['order_id', 'customer_id', 'category', 'price', 'quantity', 'order_date', 'amount', 'year', 'month', 'weekday']


In [21]:
# 8단계(미니 과제 정답 예시) — transform.py 끝에 이어 붙입니다
for cat in df['category'].unique():
    sub = df[df['category'] == cat]
    print(cat, round(sub['amount'].mean()))

print(df[df['quantity'] == 5][['order_id', 'amount']].head())
print(df.sort_values('amount', ascending=False).head(10)[['order_id', 'category', 'amount']])

의류 295678
도서 289009
가전 293831
식품 290265
    order_id    amount
1          2  670000.0
4          5  705000.0
5          6  370000.0
11        12  505000.0
53        54  360000.0
     order_id category    amount
121       122       가전  995000.0
206       207       의류  995000.0
148       149       식품  985000.0
784       785       도서  980000.0
403       404       가전  980000.0
425       426       식품  980000.0
198       199       의류  975000.0
943       944       식품  965000.0
752       753       식품  960000.0
838       839       의류  955000.0
